# Imports and client init

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import io
import csv
import pandas as pd
from collections import defaultdict
from dotenv import load_dotenv
from datetime import datetime
import requests
import uuid

from linalgo.hub.client import LinalgoClient
from linalgo.annotate.models import Corpus, Document, Annotation, Entity, AnnotatorFactory, Target
from wsd.load_data import load_data
from lineval.utils import Body
from linalgo.annotate import models

In [ ]:
load_dotenv()
token = os.getenv('LINHUB_TOKEN')
url = "https://linhub.api.linalgo.com/v1"
client = LinalgoClient(token, url)
jack_org_id = "acf7a1aa-ec18-4fa2-a981-a756bc6e6af2"
test_id = "6667052e-b464-47a9-beca-dd8df8f8c632"
jack_org = client.get_organization(jack_org_id)

# Load and import Semcor docs

In [ ]:
#loading candidates
X, y = load_data(lang='fr')

k = len(X)
X_test, y_test = X[:k], y[:k]
len(X_test), len(y_test)

In [ ]:
semcor_corpus = Corpus(name='Semcor_fr', description='The Semcor wsd corpus for French', organization=jack_org)
corpus = client.create_corpus(semcor_corpus, jack_org)

In [ ]:
target = Target()

In [ ]:
# canidates to corpus
grouped_X = defaultdict(list)
for i,row in enumerate(X_test):
    row.lemma_meaning = y_test[i]
    grouped_X[(row.lemma, row.pos)].append(row)
items = grouped_X.items()
docs = []
for g, cands in items:
    contexts = "\n".join([anno.context for anno in cands])
    doc = Document(content=contexts,
                   corpus=semcor_corpus
                   )
    doc_annos = []
    for c in cands:
        anno = Annotation(document=doc,
                          entity=c.lemma_meaning,
                          body=Body(text=c.text, context=c.context),
                          task="task",
                          annotator="none",
                          target={},
                          created=datetime.now())
        doc_annos.append(anno)
    doc.annotations = set(doc_annos)
    docs.append(doc)

semcor_corpus.documents = docs
X_docs = semcor_corpus.documents
len(X_docs)

# Adding documents

In [ ]:
client.add_documents(X_docs)

# Creating task

In [ ]:
#Generate a UID for the task
new_task_id = str(uuid.uuid4())
new_task_id

In [ ]:
exsting_task_id = "d3ce7764-eb85-4999-b965-c028f539ee33"
entities = client.get_task(exsting_task_id).entities

In [ ]:
def create_task(
    name: str,
    organization: str,
    task_id : str = str(uuid.uuid4()),
    description: str = None,
    corpus_id: str = None,
    entities: list[Entity] = [],
    ) -> None:


    serialized_entities = [entity.id for entity in entities]

    post_url = url + f"/tasks/"
    data  = {
        "id": task_id,
        "name": name,
        "slug": name,
        "organization": organization,
        "description": description,
        "entities": serialized_entities,
        "corpora": [corpus_id],
    }
    client.post(url = post_url, data= data)
    pass

In [ ]:
# create_task(name = "last_test_task",
#             organization = jack_org_id,
#             task_id = new_task_id,
#             corpus_id = semcor_corpus.id,
#             entities = entities)

In [ ]:
new_task_id

In [ ]:
task = client.get_task(new_task_id, verbose=True)

# Getting gold annotator 

In [ ]:
annotators = client.get(url = f"{url}/annotators/?page_size=1000")["results"]
len(annotators)

In [ ]:
for a in annotators:
    if "gold" in a["name"]:
        print(a)

In [ ]:
for a in annotators:
    if "gold" == a["name"] and 77 == a["owner"]:
        gold_anotator_dict = a
        print(gold_anotator_dict)

In [ ]:
gold = AnnotatorFactory.from_dict(gold_anotator_dict)
gold

# adding annotations

In [ ]:
annos = []

for doc in X_docs:
    for anno in doc.annotations:
        anno.annotator = gold
        annos.append(anno)
len(annos)

In [ ]:
annos[0].__dict__

In [ ]:
for anno in annos:
    client.create_annotations(annos)